# Week 4: Recursion, Call Stack & Memoization — PHASE 2 "First Data Structure Choices"

*📚 Data Structures & Algorithms · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Understand what recursion is and how it differs from iteration
2. Identify the **base case** and **recursive case** in a recursive function
3. Trace through the **call stack** to understand how recursive calls execute
4. Recognize the problem of **exponential blowup** in naive recursion
5. Apply **memoization** to eliminate redundant computations
6. Use `functools.lru_cache` as a convenient memoization shortcut
7. Benchmark naive vs memoized recursion and interpret the results

## 🎯 Core Mastery Connection

Memoization trades space for time — a fundamental trade-off in data structure design. This week you will see a dramatic example: naive Fibonacci is O(2^n), but adding a dictionary cache (memoization) drops it to O(n). You will benchmark both and see the difference go from seconds to microseconds. This is your first encounter with choosing to add a data structure (the cache) to solve a performance problem.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import matplotlib.pyplot as plt

import functools
import sys
import time

---
## Part 1: What Is Recursion?

**Recursion** is when a function calls **itself** to solve a smaller version of the same problem.

### Real-World Analogies

| Analogy | How It Relates |
|---|---|
| 🪆 **Russian nesting dolls** | Open a doll → find a smaller doll inside → keep opening until the smallest one |
| 📖 **Looking up a word** | You look up "happy" → definition says "feeling joy" → you look up "joy" → and so on, until you reach a word you already know |
| 🪞 **Mirrors facing each other** | Each reflection contains another reflection → but eventually the image fades (base case!) |

### The Two Essential Parts

Every recursive function needs:

1. **Base case** — the simplest version of the problem that we can answer directly (stops the recursion)
2. **Recursive case** — break the problem into a smaller piece and call the function again

**Figure 1.1** — A simple countdown using recursion

In [ ]:
def countdown(n):
    """Count down from n to 0 using recursion."""
    if n < 0:          # Base case: stop when n is negative
        print("Go!")
        return
    print(n)           # Do something with current value
    countdown(n - 1)   # Recursive case: call with smaller n

countdown(5)

**Figure 1.2** — Comparing recursion to a simple loop

In [ ]:
# The same countdown with a loop (iterative version)
def countdown_loop(n):
    for i in range(n, -1, -1):
        print(i)
    print("Go!")

print("=== Recursive ===")
countdown(3)

print("\n=== Iterative ===")
countdown_loop(3)

---
## Part 2: Classic Example — Factorial

The **factorial** of a number `n` (written `n!`) is:

```
5! = 5 × 4 × 3 × 2 × 1 = 120
```

Recursive definition:
- **Base case:** `0! = 1`
- **Recursive case:** `n! = n × (n-1)!`

**Figure 2.1** — Recursive factorial implementation

In [ ]:
def factorial(n):
    """Calculate n! recursively."""
    if n == 0:             # Base case
        return 1
    return n * factorial(n - 1)  # Recursive case

# Test it
for i in range(8):
    print(f"{i}! = {factorial(i)}")

**Figure 2.2** — Tracing the factorial calls step by step

In [ ]:
def factorial_traced(n, depth=0):
    """Factorial with indented trace output."""
    indent = "  " * depth
    print(f"{indent}factorial({n}) called")
    
    if n == 0:
        print(f"{indent}factorial(0) = 1  [base case]")
        return 1
    
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}factorial({n}) = {n} * {result // n} = {result}")
    return result

print("Tracing factorial(4):")
print("="*40)
answer = factorial_traced(4)
print("="*40)
print(f"Final answer: {answer}")

---
## Part 3: Visualizing the Call Stack

When a function calls itself, Python keeps track of each call on the **call stack**.

Think of it like a stack of plates:
- Each new function call adds a plate on top
- When a function returns, its plate is removed
- The bottom plate is always the first call

```
Call Stack for factorial(4):

  factorial(0) = 1        ← top (returns first)
  factorial(1) = 1*1 = 1
  factorial(2) = 2*1 = 2
  factorial(3) = 3*2 = 6
  factorial(4) = 4*6 = 24 ← bottom (returns last)
```

**Figure 3.1** — Simulating the call stack with a list

In [ ]:
def factorial_stack_viz(n):
    """Visualize the call stack for factorial."""
    stack = []
    
    # Push phase: build up the stack
    print(">>> PUSHING onto stack (going deeper):")
    for i in range(n, -1, -1):
        stack.append(i)
        print(f"   Stack: {stack}")
    
    # Pop phase: resolve from top
    print("\n<<< POPPING from stack (returning results):")
    result = 1
    while stack:
        value = stack.pop()
        if value == 0:
            print(f"   Popped 0 -> base case = 1")
        else:
            result *= value
            print(f"   Popped {value} -> running product = {result}")
    
    return result

answer = factorial_stack_viz(5)
print(f"\nFinal result: {answer}")

**Figure 3.2** — What happens when there is no base case? (RecursionError)

In [ ]:
import sys
print(f"Python's default recursion limit: {sys.getrecursionlimit()}")

def infinite_recursion(n):
    """WARNING: This function has no base case!"""
    return infinite_recursion(n + 1)

# Let's catch the error
try:
    infinite_recursion(1)
except RecursionError as e:
    print(f"\nRecursionError caught: {e}")
    print("\nLesson: Always define a base case!")

---
## Part 4: Fibonacci — The Problem with Naive Recursion

The **Fibonacci sequence**: 0, 1, 1, 2, 3, 5, 8, 13, 21, ...

Each number is the sum of the two before it:
- **Base cases:** `fib(0) = 0`, `fib(1) = 1`
- **Recursive case:** `fib(n) = fib(n-1) + fib(n-2)`

**Figure 4.1** — Naive recursive Fibonacci

In [ ]:
def fib_naive(n):
    """Calculate Fibonacci number recursively (naive)."""
    if n <= 1:        # Base cases
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)  # Two recursive calls!

# Test with small values
for i in range(12):
    print(f"fib({i:2d}) = {fib_naive(i)}")

**Figure 4.2** — Counting how many function calls are made

In [ ]:
call_count = 0

def fib_counted(n):
    """Fibonacci that counts how many times it's called."""
    global call_count
    call_count += 1
    
    if n <= 1:
        return n
    return fib_counted(n - 1) + fib_counted(n - 2)

# See how calls explode
print(f"{'n':>4}  {'fib(n)':>10}  {'calls':>10}")
print("-" * 30)
for n in range(1, 26):
    call_count = 0
    result = fib_counted(n)
    print(f"{n:4d}  {result:10d}  {call_count:10d}")

### The Call Tree Problem

For `fib(5)`, the call tree looks like:

```
                    fib(5)
                   /      \
              fib(4)      fib(3)
             /    \       /    \
         fib(3)  fib(2) fib(2) fib(1)
         /   \    /  \   /  \
      fib(2) fib(1) ... ...  ...
```

Notice how `fib(3)` is calculated **twice**, `fib(2)` is calculated **three times**, etc.

This is **exponential blowup**: the number of calls roughly doubles with each increase in `n`.

---
## Part 5: Memoization — Remember What You Already Computed

**Memoization** = storing results of expensive function calls so we don't repeat them.

| Without Memoization | With Memoization |
|---|---|
| Recalculates `fib(3)` every time it's needed | Calculates `fib(3)` once, stores the result |
| Exponential time: O(2^n) | Linear time: O(n) |
| `fib(40)` takes seconds | `fib(40)` is instant |

**Figure 5.1** — Memoized Fibonacci using a dictionary cache

In [ ]:
def fib_memo(n, cache=None):
    """Fibonacci with memoization using a dictionary."""
    if cache is None:
        cache = {}       # Create cache on first call
    
    if n in cache:       # Already computed? Return it!
        return cache[n]
    
    if n <= 1:           # Base cases
        return n
    
    # Compute, store in cache, then return
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]

# Now we can compute much larger values instantly
for n in [10, 20, 30, 50, 100]:
    print(f"fib({n:3d}) = {fib_memo(n)}")

**Figure 5.2** — Seeing the cache in action

In [ ]:
def fib_memo_verbose(n, cache=None, depth=0):
    """Memoized Fibonacci with verbose output."""
    if cache is None:
        cache = {}
    
    indent = "  " * depth
    
    if n in cache:
        print(f"{indent}fib({n}) -> CACHE HIT = {cache[n]}")
        return cache[n]
    
    if n <= 1:
        print(f"{indent}fib({n}) -> base case = {n}")
        return n
    
    print(f"{indent}fib({n}) -> computing...")
    cache[n] = fib_memo_verbose(n - 1, cache, depth + 1) + fib_memo_verbose(n - 2, cache, depth + 1)
    print(f"{indent}fib({n}) -> STORED = {cache[n]}")
    return cache[n]

print("Tracing fib_memo(6):")
print("=" * 50)
result = fib_memo_verbose(6)
print("=" * 50)
print(f"Result: {result}")

---
## Part 6: The Easy Way — `functools.lru_cache`

Python provides a built-in **decorator** that adds memoization automatically.

LRU = "Least Recently Used" — it keeps the most recent results and discards old ones if the cache gets too big.

**Figure 6.1** — Using `@lru_cache` for automatic memoization

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)   # None = unlimited cache size
def fib_lru(n):
    """Fibonacci with automatic memoization via lru_cache."""
    if n <= 1:
        return n
    return fib_lru(n - 1) + fib_lru(n - 2)

# Works great for large values
for n in [10, 50, 100, 200, 500]:
    print(f"fib({n:3d}) = {fib_lru(n)}")

# Check cache statistics
print(f"\nCache info: {fib_lru.cache_info()}")

**Figure 6.2** — Comparing: manual cache vs `lru_cache` vs naive

In [ ]:
# All three give the same result
n = 20
print(f"Naive:     fib({n}) = {fib_naive(n)}")
print(f"Manual:    fib({n}) = {fib_memo(n)}")
print(f"lru_cache: fib({n}) = {fib_lru(n)}")
print("\nAll three produce the same answer.")
print("The difference is SPEED, which we'll measure next.")

---
## Part 7: Benchmarking — Naive vs Memoized

Let's measure how long each approach takes and see the difference with real numbers.

> **🔮 Predict first, then measure. Does reality match your prediction?** Before running: predict how naive fib(30) will compare to memoized fib(30). What speedup factor do you expect?

**Figure 7.1** — Timing naive Fibonacci for increasing values of n

In [ ]:
import time

def time_function(func, n):
    """Time how long func(n) takes. Returns time in seconds."""
    start = time.perf_counter()
    result = func(n)
    elapsed = time.perf_counter() - start
    return elapsed, result

# Benchmark naive Fibonacci
print("Naive Fibonacci Timing")
print(f"{'n':>4}  {'time (sec)':>12}  {'result':>12}")
print("-" * 35)

naive_times = []
test_ns_naive = list(range(5, 36, 5))

for n in test_ns_naive:
    t, r = time_function(fib_naive, n)
    naive_times.append(t)
    print(f"{n:4d}  {t:12.6f}  {r:12d}")

**Figure 7.2** — Timing memoized Fibonacci for much larger values

In [ ]:
# Benchmark memoized Fibonacci
print("Memoized Fibonacci Timing")
print(f"{'n':>6}  {'time (sec)':>12}")
print("-" * 25)

memo_times = []
test_ns_memo = [10, 50, 100, 200, 500, 800]

for n in test_ns_memo:
    # Fresh cache each time for fair comparison
    cache = {}
    t, r = time_function(lambda x: fib_memo(x, {}), n)
    memo_times.append(t)
    print(f"{n:6d}  {t:12.8f}")

print("\nNotice: even fib(800) is nearly instant with memoization!")

**Figure 7.3** — Side-by-side comparison on the same values

In [ ]:
# Compare both on the same n values
compare_ns = [5, 10, 15, 20, 25, 30]
naive_data = []
memo_data = []

print(f"{'n':>4}  {'Naive (sec)':>14}  {'Memoized (sec)':>14}  {'Speedup':>10}")
print("-" * 50)

for n in compare_ns:
    t_naive, _ = time_function(fib_naive, n)
    t_memo, _ = time_function(lambda x: fib_memo(x, {}), n)
    
    naive_data.append(t_naive)
    memo_data.append(t_memo)
    
    speedup = t_naive / t_memo if t_memo > 0 else float('inf')
    print(f"{n:4d}  {t_naive:14.8f}  {t_memo:14.8f}  {speedup:10.1f}x")

**Figure 7.4** — Plotting naive vs memoized Fibonacci performance

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Naive Fibonacci growth
ax1.plot(test_ns_naive, naive_times, 'ro-', linewidth=2, markersize=8)
ax1.set_xlabel('n', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Naive Fibonacci: Exponential Growth', fontsize=14)
ax1.grid(True, alpha=0.3)

# Plot 2: Comparison (log scale)
ax2.plot(compare_ns, naive_data, 'ro-', label='Naive', linewidth=2, markersize=8)
ax2.plot(compare_ns, memo_data, 'gs-', label='Memoized', linewidth=2, markersize=8)
ax2.set_xlabel('n', fontsize=12)
ax2.set_ylabel('Time (seconds, log scale)', fontsize=12)
ax2.set_title('Naive vs Memoized Fibonacci', fontsize=14)
ax2.set_yscale('log')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Takeaway: Naive recursion grows exponentially.")
print("Memoization keeps it linear by avoiding redundant work.")

---
## Part 8: More Recursion Examples

**Figure 8.1** — Sum of a list using recursion

In [ ]:
def recursive_sum(lst):
    """Sum all elements in a list recursively."""
    if len(lst) == 0:           # Base case: empty list
        return 0
    return lst[0] + recursive_sum(lst[1:])  # First element + sum of rest

numbers = [3, 7, 1, 9, 4]
print(f"List: {numbers}")
print(f"Recursive sum: {recursive_sum(numbers)}")
print(f"Built-in sum:  {sum(numbers)}")

**Figure 8.2** — Reversing a string recursively

In [ ]:
def reverse_string(s):
    """Reverse a string using recursion."""
    if len(s) <= 1:            # Base case
        return s
    return reverse_string(s[1:]) + s[0]  # Reverse the rest, then add first char

test_words = ["hello", "Python", "recursion", "abcdef"]
for word in test_words:
    print(f"{word:>12} -> {reverse_string(word)}")

**Figure 8.3** — Calculating power (x^n) recursively

In [ ]:
def power(x, n):
    """Calculate x raised to the power n recursively."""
    if n == 0:                 # Base case: anything to the 0 is 1
        return 1
    return x * power(x, n - 1) # x^n = x * x^(n-1)

print(f"2^10 = {power(2, 10)}")
print(f"3^5  = {power(3, 5)}")
print(f"5^3  = {power(5, 3)}")

---
## Part 9: Common Errors with Recursion

**Figure 9.1** — Error: Forgetting the base case

In [ ]:
# ERROR: No base case!
def bad_factorial(n):
    return n * bad_factorial(n - 1)  # Never stops!

try:
    bad_factorial(5)
except RecursionError:
    print("RecursionError: No base case means infinite recursion!")
    print("Fix: Add 'if n == 0: return 1' at the beginning.")

**Figure 9.2** — Error: Base case doesn't match the recursive step

In [ ]:
# ERROR: Base case never reached for negative inputs
def bad_countdown(n):
    if n == 0:                # Only stops at exactly 0
        return
    print(n)
    bad_countdown(n - 2)      # Skips by 2, might miss 0!

print("With n=5 (odd number), this skips past 0:")
try:
    import sys
    sys.setrecursionlimit(50)  # Set low limit to catch it faster
    bad_countdown(5)
except RecursionError:
    print("...RecursionError! 5 -> 3 -> 1 -> -1 -> never hits 0")
    print("Fix: Use 'if n <= 0' instead of 'if n == 0'")
finally:
    sys.setrecursionlimit(1000)  # Reset to default

**Figure 9.3** — Error: Mutable default argument trap

In [ ]:
# CAREFUL: Mutable default arguments are shared between calls!
def buggy_collect(n, result=[]):  # BAD: list is shared!
    if n == 0:
        return result
    result.append(n)
    return buggy_collect(n - 1, result)

print("First call:", buggy_collect(3))   # [3, 2, 1] - looks fine
print("Second call:", buggy_collect(3))  # [3, 2, 1, 3, 2, 1] - BUG!

print("\nFix: Use None as default and create a new list inside:")

def fixed_collect(n, result=None):
    if result is None:
        result = []            # Fresh list each time
    if n == 0:
        return result
    result.append(n)
    return fixed_collect(n - 1, result)

print("First call:", fixed_collect(3))
print("Second call:", fixed_collect(3))

---
## 🌉 Bridge to Next Week

This week we learned **recursion** — a powerful technique where a function calls itself to solve smaller subproblems.

Key takeaways:
- Every recursive function needs a **base case** and a **recursive case**
- Naive recursion can be **exponentially slow** due to redundant calculations
- **Memoization** stores previously computed results to avoid this
- Python's `@lru_cache` makes memoization easy

**Next week**, we'll explore **Searching** — how to find elements in a collection efficiently. We'll compare **linear search** (checking every element) with **binary search** (divide and conquer), and discover how binary search uses a recursive-like strategy to achieve O(log n) performance.

---
## 🎢 Exercises

Complete each exercise in the code cell below its description. Make sure to **run** each cell before submitting.

> **🔮 Predict first, then measure. Does reality match your prediction?** For every exercise that involves timing or complexity analysis, make your prediction BEFORE running the code.

### Easy

**EX1 — Recursive Sum of Digits**

Write a recursive function `sum_digits(n)` that returns the sum of the digits of a positive integer `n`.

**Expected Output:**
```
sum_digits(123) = 6
sum_digits(9999) = 36
sum_digits(5) = 5
```

<details><summary>💡 Hint</summary>
The last digit of n is `n % 10`. The remaining digits are `n // 10`. Base case: when n is a single digit (n < 10).
</details>

In [ ]:
# ✏️ [EX1] Your code here


**EX2 — Recursive Countdown String**

Write a recursive function `countdown_str(n)` that returns a string like `"5-4-3-2-1-Go!"`.

**Expected Output:**
```
countdown_str(5) = "5-4-3-2-1-Go!"
countdown_str(3) = "3-2-1-Go!"
countdown_str(1) = "1-Go!"
```

<details><summary>💡 Hint</summary>
Base case: when n == 0, return "Go!". Recursive case: return `str(n) + "-" + countdown_str(n - 1)`.
</details>

In [ ]:
# ✏️ [EX2] Your code here


**EX3 — Recursive Length of a List**

Write a recursive function `rec_len(lst)` that returns the length of a list without using `len()`.

**Expected Output:**
```
rec_len([]) = 0
rec_len([10, 20, 30]) = 3
rec_len([1, 2, 3, 4, 5, 6]) = 6
```

<details><summary>💡 Hint</summary>
Base case: empty list has length 0. Recursive case: 1 + rec_len(lst[1:]).
</details>

In [ ]:
# ✏️ [EX3] Your code here


**EX4 — Recursive Multiplication**

Write a recursive function `multiply(a, b)` that multiplies two positive integers using only addition (no `*` operator).

**Expected Output:**
```
multiply(3, 4) = 12
multiply(5, 6) = 30
multiply(7, 1) = 7
```

<details><summary>💡 Hint</summary>
Think of 3 × 4 as 3 + 3 + 3 + 3. Base case: when b == 0, return 0. Recursive case: a + multiply(a, b - 1).
</details>

In [ ]:
# ✏️ [EX4] Your code here


### Medium

**EX5 — Recursive Palindrome Check**

Write a recursive function `is_palindrome(s)` that returns `True` if a string reads the same forwards and backwards.

**Expected Output:**
```
is_palindrome("racecar") = True
is_palindrome("hello") = False
is_palindrome("madam") = True
is_palindrome("a") = True
```

<details><summary>💡 Hint</summary>
Compare the first and last characters. If they match, recursively check the substring without the first and last characters. Base case: string of length 0 or 1 is always a palindrome.
</details>

In [ ]:
# ✏️ [EX5] Your code here


**EX6 — Recursive Flatten**

Write a recursive function `flatten(lst)` that takes a nested list and returns a flat list.

**Expected Output:**
```
flatten([1, [2, 3], [4, [5, 6]]]) = [1, 2, 3, 4, 5, 6]
flatten([[1, 2], [3, [4, [5]]]]) = [1, 2, 3, 4, 5]
```

<details><summary>💡 Hint</summary>
Loop through items. If an item is a list, recursively flatten it and extend your result. If it's not a list, just append it.
</details>

In [ ]:
# ✏️ [EX6] Your code here


**EX7 — Memoized Tribonacci**

The Tribonacci sequence is like Fibonacci but adds the last **three** numbers: `trib(n) = trib(n-1) + trib(n-2) + trib(n-3)`, with `trib(0)=0, trib(1)=0, trib(2)=1`.

Write a memoized version `trib(n)` and compute `trib(30)`.

**Expected Output:**
```
trib(10) = 149
trib(20) = 66012
trib(30) = 29249425
```

<details><summary>💡 Hint</summary>
Use either a dictionary cache (like fib_memo) or @lru_cache. Three base cases needed.
</details>

In [ ]:
# ✏️ [EX7] Your code here


**EX8 — Recursive Maximum**

Write a recursive function `rec_max(lst)` that finds the maximum value in a list without using `max()`.

**Expected Output:**
```
rec_max([3, 7, 2, 9, 1]) = 9
rec_max([5]) = 5
rec_max([10, 20, 15, 25, 5]) = 25
```

<details><summary>💡 Hint</summary>
Base case: list with one element, return that element. Recursive case: compare first element with rec_max of the rest.
</details>

In [ ]:
# ✏️ [EX8] Your code here


**EX9 — Recursive Power (Fast)**

Write `fast_power(x, n)` that uses the insight: if n is even, `x^n = (x^(n/2))^2`. This reduces the number of multiplications.

**Expected Output:**
```
fast_power(2, 10) = 1024
fast_power(3, 13) = 1594323
fast_power(5, 0) = 1
```

<details><summary>💡 Hint</summary>
If n is 0, return 1. If n is even: half = fast_power(x, n // 2), return half * half. If n is odd: return x * fast_power(x, n - 1).
</details>

In [ ]:
# ✏️ [EX9] Your code here


**EX10 — Count Occurrences**

Write a recursive function `count_occurrences(lst, target)` that counts how many times `target` appears in `lst`.

**Expected Output:**
```
count_occurrences([1, 2, 3, 2, 4, 2], 2) = 3
count_occurrences([5, 5, 5, 5], 5) = 4
count_occurrences([1, 2, 3], 9) = 0
```

<details><summary>💡 Hint</summary>
Base case: empty list returns 0. Recursive case: check if first element equals target (1 or 0) + count in the rest of the list.
</details>

In [ ]:
# ✏️ [EX10] Your code here


### Challenge

**EX11 — Tower of Hanoi**

Write a recursive function `hanoi(n, source, target, auxiliary)` that prints the steps to solve the Tower of Hanoi puzzle with `n` disks.

**Expected Output (n=3):**
```
Move disk 1 from A to C
Move disk 2 from A to B
Move disk 1 from C to B
Move disk 3 from A to C
Move disk 1 from B to A
Move disk 2 from B to C
Move disk 1 from A to C
```

<details><summary>💡 Hint</summary>
To move n disks from source to target: (1) move n-1 disks from source to auxiliary, (2) move disk n from source to target, (3) move n-1 disks from auxiliary to target. Base case: n == 1, just move the single disk.
</details>

In [ ]:
# ✏️ [EX11] Your code here


**EX12 — Memoized Grid Paths**

Write a memoized function `grid_paths(m, n)` that counts how many unique paths exist from the top-left to the bottom-right of an `m x n` grid, moving only right or down.

**Expected Output:**
```
grid_paths(2, 2) = 2
grid_paths(3, 3) = 6
grid_paths(10, 10) = 48620
grid_paths(20, 20) = 35345263800
```

<details><summary>💡 Hint</summary>
At any cell, you can go right or down: `grid_paths(m, n) = grid_paths(m-1, n) + grid_paths(m, n-1)`. Base case: if m == 1 or n == 1, there's only 1 path. Use @lru_cache or a dictionary.
</details>

In [ ]:
# ✏️ [EX12] Your code here


---
## 📮 Submission

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 1: Fill in your info below, then run this cell
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STUDENT_ID    = ""     # e.g. "2024001234"
STUDENT_NAME  = ""     # e.g. "Ahmet Y\u0131lmaz"
STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"
CLASS_CODE    = ""     # code given in class
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Don't change anything below this line
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re as _re
_errors = []
if not _re.match(r"^\d{6,12}$", STUDENT_ID):
    _errors.append("\u274c Student ID must be 6-12 digits")
if len(STUDENT_NAME.strip().split()) < 2:
    _errors.append("\u274c Enter first and last name")
if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:
    _errors.append("\u274c Use your @istun.edu.tr email")
if len(CLASS_CODE.strip()) < 4:
    _errors.append("\u274c Invalid class code")
if _errors:
    for _e in _errors:
        print(_e)
    print("\n\u26a0\ufe0f  Fix the errors above and run this cell again.")
else:
    print(f"\u2705 Info OK \u2014 {STUDENT_NAME} ({STUDENT_ID})")
    print(f"   {STUDENT_EMAIL}")
    print(f"\n\ud83d\udc49 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📮 STEP 2: Run this cell to submit
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json, re, os, urllib.request
WEEK = "Week_04"
URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"
try:
    _sid = STUDENT_ID.strip()
    _sname = STUDENT_NAME.strip()
    _semail = STUDENT_EMAIL.strip().lower()
    _scode = CLASS_CODE.strip().upper()
except NameError:
    raise SystemExit("\u274c Run the cell above first to set your info!")
if not _sid or not _sname or not _semail or not _scode:
    raise SystemExit("\u274c Run the cell above first \u2014 some fields are empty.")
_answers = {}
try:
    _ipy = get_ipython()
    _hist = _ipy.history_manager.get_range(output=False)
    for _sess, _line, _src in _hist:
        _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
        if _m:
            _ex_id = "ex" + _m.group(1)
            _lines = _src.split("\n")
            _clean = "\n".join(_lines[1:]).strip()
            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
except Exception:
    pass
if not _answers:
    try:
        for _src in In:
            if not _src: continue
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
    except NameError:
        pass
if not _answers:
    _nb_path = None
    try:
        _nb_path = __vsc_ipynb_file__
    except NameError:
        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]
        if len(_candidates) == 1: _nb_path = _candidates[0]
    if _nb_path and os.path.exists(str(_nb_path)):
        with open(str(_nb_path), "r", encoding="utf-8") as _f:
            _nb = json.load(_f)
        for _cell in _nb["cells"]:
            if _cell["cell_type"] != "code": continue
            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
            _m = re.match(r"#\s*\u270f\ufe0f\s*\[EX(\w+)\]", _src)
            if _m:
                _ex_id = "ex" + _m.group(1)
                _lines = _src.split("\n")
                _clean = "\n".join(_lines[1:]).strip()
                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}
print(f"\ud83d\udcdd Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")
if not _answers:
    print("\n\u26a0\ufe0f  No exercise answers found!")
    print("Make sure you RAN all exercise cells before submitting.")
    raise SystemExit()
_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "dsa-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")
print("\ud83d\udce1 Submitting...")
try:
    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
    _resp = urllib.request.urlopen(_req, timeout=30)
    _result = json.loads(_resp.read().decode())
    if _result.get("success"):
        print(f"\n\u2705 {_result['message']}")
        print("\ud83d\udce7 Check your email for confirmation.")
    else:
        print(f"\n\u274c {_result.get('message', 'Submission failed')}")
except Exception as _e:
    try:
        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")
        urllib.request.urlopen(_req, timeout=10)
    except: pass
    print(f"\n\u26a0\ufe0f  Request sent \u2014 check your email for confirmation.")
    print(f"(If no email arrives, try again or contact your instructor)")